In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
train_df

In [ ]:
train_df.describe(include='object')

In [ ]:
test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
test_df

In [ ]:
test_df.describe(include='object')

In [ ]:
train_df['input'] = 'KEYWORD: ' + train_df.keyword.fillna('NaN') + '; LOCATION: ' + train_df.location.fillna('NaN') + '; TEXT1: ' + train_df.text.fillna('NaN')

In [ ]:
test_df['input'] = 'KEYWORD: ' + test_df.keyword.fillna('NaN') + '; LOCATION: ' + test_df.location.fillna('NaN') + '; TEXT1: ' + test_df.text.fillna('NaN')

# all the inputs

In [ ]:
train_df.input.head()

## test input

In [ ]:
test_df.input.head()

In [ ]:
train_df.head()

In [ ]:
test_df.head()

# Tokenization

In [ ]:
from datasets import Dataset,DatasetDict

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)


In [ ]:
model_nm = 'microsoft/deberta-v3-small'
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)

In [ ]:
def tok_func(x): return tokz(x["input"])

In [ ]:
train_tok_ds = train_ds.map(tok_func, batched=True)

In [ ]:
test_tok_ds = test_ds.map(tok_func, batched=True)

In [ ]:
print(train_tok_ds)


# Renaming column

In [ ]:
train_tok_ds = train_tok_ds.rename_columns({'target':'labels'})

# DataSetDict

In [ ]:
dds = train_tok_ds.train_test_split(0.01, seed=42)
dds

# Training

In [ ]:
from transformers import TrainingArguments,Trainer

In [ ]:
# bs = 128  # out-of-mem
bs = 32
# epochs = 4
# epochs = 1  # LB 0.82592
# epochs = 2  # LB 0.82470
# epochs = 3  # LB 0.

epochs = 1  # LB 0.

lr = 8e-5

In [ ]:
# args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, lr_scheduler_type='cosine', fp16=True,
#     evaluation_strategy="epoch", per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
#     num_train_epochs=epochs, weight_decay=0.01, report_to='none')

args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, 
                         lr_scheduler_type='cosine', 
                         fp16=True,
    eval_strategy="epoch", 
                         per_device_train_batch_size=bs, 
                         per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, 
                         weight_decay=0.01, 
                         report_to='none')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_nm, num_labels=2)

trainer = Trainer(
    model, args, 
    train_dataset=dds['train'], 
    eval_dataset=dds['test'],
                  tokenizer=tokz, 
#     compute_metrics=my_metric
)
# model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=1)
# trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'],
#                   tokenizer=tokz, compute_metrics=corr_d)

In [ ]:
trainer.train();

In [ ]:
preds = trainer.predict(test_tok_ds)
preds

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)
class_preds

In [ ]:
preds

In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub

In [ ]:
sub['target'] = class_preds
sub

In [ ]:
# 1 epoch = LB 0.82592  'microsoft/deberta-v3-small' 1 epochs
sub.to_csv(f'submission.csv', index=False)